# Data Preparation Validation

This notebook validates the outputs produced by
`src/prepare_data.py`.

The transformation logic is kept in a Python script so that the data
preparation process is reproducible. This notebook is used to inspect
and verify the resulting analytical dataset.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
REPORTS_DIR = PROJECT_ROOT / "reports"

POLICY_PATH = (
    PROCESSED_DATA_DIR / "policy_analytics.csv"
)

AUDIT_PATH = (
    INTERIM_DATA_DIR / "claim_count_audit.csv"
)

print(f"Policy file exists: {POLICY_PATH.exists()}")
print(f"Audit file exists: {AUDIT_PATH.exists()}")

Policy file exists: True
Audit file exists: True


In [2]:
policy = pd.read_csv(POLICY_PATH)
audit = pd.read_csv(AUDIT_PATH)

print(f"Policy analytics shape: {policy.shape}")
print(f"Claim-count audit shape: {audit.shape}")

Policy analytics shape: (678013, 39)
Claim-count audit shape: (678019, 7)


In [3]:
display(policy.head())

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,ValidPolicyID,ValidClaimCount,ValidExposure,ValidVehicleAge,ValidDriverAge,ValidBonusMalus,ValidDensity,CorePolicyRecordValid,AnyPredictorValidityIssue,SeverityRowCount,SeverityTotalClaimAmount,SeverityAverageClaimAmount,SeverityMinimumClaimAmount,SeverityMaximumClaimAmount,HasClaim,HasSeverityRecord,ClaimCountDifference,ClaimCountMatch,SeverityCompletenessStatus,AnnualizedClaimFrequency,ObservedAverageClaimAmount,CompleteTotalClaimAmount,CompleteAverageClaimAmount,CompletePurePremium,FrequencyModelEligible,SeverityModelEligible,PurePremiumModelEligible
0,1,1,0.100,D,5,0,55,50,B12,'Regular',1217,R82,True,True,True,True,True,True,True,True,False,0,0.000,NaN,NaN,NaN,1,0,1,False,claim_without_severity,10.000,NaN,NaN,NaN,NaN,True,False,False
1,3,1,0.770,D,5,0,55,50,B12,'Regular',1217,R82,True,True,True,True,True,True,True,True,False,0,0.000,NaN,NaN,NaN,1,0,1,False,claim_without_severity,1.299,NaN,NaN,NaN,NaN,True,False,False
2,5,1,0.750,B,6,2,52,50,B12,'Diesel',54,R22,True,True,True,True,True,True,True,True,False,0,0.000,NaN,NaN,NaN,1,0,1,False,claim_without_severity,1.333,NaN,NaN,NaN,NaN,True,False,False
3,10,1,0.090,B,7,0,46,50,B12,'Diesel',76,R72,True,True,True,True,True,True,True,True,False,0,0.000,NaN,NaN,NaN,1,0,1,False,claim_without_severity,11.111,NaN,NaN,NaN,NaN,True,False,False
4,11,1,0.840,B,7,0,46,50,B12,'Diesel',76,R72,True,True,True,True,True,True,True,True,False,0,0.000,NaN,NaN,NaN,1,0,1,False,claim_without_severity,1.190,NaN,NaN,NaN,NaN,True,False,False


In [4]:
status_summary = (
    policy["SeverityCompletenessStatus"]
    .value_counts(dropna=False)
    .rename_axis("Status")
    .reset_index(name="PolicyCount")
)

status_summary["Percentage"] = (
    status_summary["PolicyCount"]
    / len(policy)
    * 100
)

display(status_summary)

,Status,PolicyCount,Percentage
0,no_claim_no_severity,643953,94.976
1,matched_positive_claims,24943,3.679
2,claim_without_severity,9116,1.345
3,fewer_severity_rows_than_claimnb,1,0.000


In [5]:
audit_summary = (
    audit["AuditStatus"]
    .value_counts(dropna=False)
    .rename_axis("Status")
    .reset_index(name="PolicyCount")
)

audit_summary["Percentage"] = (
    audit_summary["PolicyCount"]
    / len(audit)
    * 100
)

display(audit_summary)

,Status,PolicyCount,Percentage
0,no_claim_no_severity,643953,94.976
1,matched_positive_claims,24943,3.679
2,claim_without_severity,9116,1.345
3,severity_without_frequency,6,0.001
4,fewer_severity_rows_than_claimnb,1,0.000


In [6]:
validation_results = {
    "Policy IDs are unique": policy["IDpol"].is_unique,
    "All exposure values are positive": (
        policy["Exposure"] > 0
    ).all(),
    "All claim counts are non-negative": (
        policy["ClaimNb"] >= 0
    ).all(),
    "Annualized frequency is non-negative": (
        policy["AnnualizedClaimFrequency"] >= 0
    ).all(),
    "No audit rows are unclassified": (
        audit["AuditStatus"] != "unclassified"
    ).all(),
}

validation_table = pd.DataFrame(
    {
        "Validation": validation_results.keys(),
        "Passed": validation_results.values(),
    }
)

display(validation_table)

,Validation,Passed
0,Policy IDs are unique,True
1,All exposure values are positive,True
2,All claim counts are non-negative,True
3,Annualized frequency is non-negative,True
4,No audit rows are unclassified,True


In [7]:
incomplete_policies = policy.loc[
    ~policy["ClaimCountMatch"],
    [
        "IDpol",
        "ClaimNb",
        "SeverityRowCount",
        "ClaimCountDifference",
        "SeverityTotalClaimAmount",
        "CompleteTotalClaimAmount",
        "SeverityCompletenessStatus",
    ],
]

display(incomplete_policies.head(20))

,IDpol,ClaimNb,SeverityRowCount,ClaimCountDifference,SeverityTotalClaimAmount,CompleteTotalClaimAmount,SeverityCompletenessStatus
0,1,1,0,1,0.000,NaN,claim_without_severity
1,3,1,0,1,0.000,NaN,claim_without_severity
2,5,1,0,1,0.000,NaN,claim_without_severity
3,10,1,0,1,0.000,NaN,claim_without_severity
4,11,1,0,1,0.000,NaN,claim_without_severity
5,13,1,0,1,0.000,NaN,claim_without_severity
6,15,1,0,1,0.000,NaN,claim_without_severity
7,17,1,0,1,0.000,NaN,claim_without_severity
8,18,1,0,1,0.000,NaN,claim_without_severity
9,21,1,0,1,0.000,NaN,claim_without_severity


In [8]:
complete_zero_claim_policies = policy.loc[
    (policy["ClaimNb"] == 0)
    & policy["ClaimCountMatch"],
    [
        "IDpol",
        "ClaimNb",
        "SeverityRowCount",
        "CompleteTotalClaimAmount",
        "CompletePurePremium",
    ],
]

display(complete_zero_claim_policies.head())

,IDpol,ClaimNb,SeverityRowCount,CompleteTotalClaimAmount,CompletePurePremium
9387,24952,0,0,0.000,0.000
9388,24953,0,0,0.000,0.000
9389,24955,0,0,0.000,0.000
9390,24956,0,0,0.000,0.000
9391,24958,0,0,0.000,0.000


In [9]:
model_samples = pd.DataFrame(
    {
        "ModelSample": [
            "Frequency model",
            "Severity model",
            "Pure premium model",
        ],
        "EligiblePolicies": [
            int(
                policy[
                    "FrequencyModelEligible"
                ].sum()
            ),
            int(
                policy[
                    "SeverityModelEligible"
                ].sum()
            ),
            int(
                policy[
                    "PurePremiumModelEligible"
                ].sum()
            ),
        ],
    }
)

model_samples["Percentage"] = (
    model_samples["EligiblePolicies"]
    / len(policy)
    * 100
)

display(model_samples)

,ModelSample,EligiblePolicies,Percentage
0,Frequency model,678013,100.000
1,Severity model,24943,3.679
2,Pure premium model,668896,98.655
